### Notes
- After connecting, run Install Dependencies once.
- Check all boxes on your very first run ever to install ComfyUI to Google Drive.
- Uncheck `UPDATE_COMFY_UI`, and `USE_COMFYUI_MANAGER` after your very first run if `USE_GOOGLE_DRIVE` is checked.

### Install Dependencies

In [ ]:
# @title  {"form-width":"20%"}

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFY_UI = False  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = False  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers torch==2.11.0 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128

if OPTIONS['USE_COMFYUI_MANAGER']:
  %cd custom_nodes

  ### Install custom nodes here
  !git clone https://github.com/MoonGoblinDev/Civicomfy.git

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
  ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
  ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat

  ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
  %cd ComfyUI-Manager
  !git pull

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
  !echo -= Install custom nodes dependencies =-
  !pip install GitPython
  !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies


### Download Resources

In [ ]:
# Diffusion Model
!wget -c "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors?download=true" -O "./models/checkpoints/anima-base-v1.0.safetensors"

# Text Encoder...
!wget -c "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors?download=true" -O "./models/text_encoders/qwen_3_06b_base.safetensors"

# VAE
!wget -c "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors?download=true" -O "./models/vae/qwen_image_vae.safetensors"

# Turbo LoRAs (This won't be needed after the release of the official Anima Turbo model)
# Including v0.1 as well - some people prefer 0.1 to 0.2, so give that option.
!wget -c "https://huggingface.co/circlestone-labs/Anima-Official-LoRAs/resolve/main/anima-turbo-lora-v0.2.safetensors?download=true" -O "./models/loras/anima-turbo-lora-v0.2.safetensors"
!wget -c "https://huggingface.co/circlestone-labs/Anima-Official-LoRAs/resolve/main/anima-turbo-lora-v0.1.safetensors?download=true" -O "./models/loras/anima-turbo-lora-v0.1.safetensors"


### Run ComfyUI

In [ ]:
# @title  {"form-width":"20%"}
### It appears that the standard google.colab.output is not working - you get a 403 error when attempting to connect.
### For now you'll need to use a tunnel. CloudflareD or LOCALTUNNEL both should work - pick one.
USE_CLOUDFLARED = False  #@param {type:"boolean"}
USE_LOCALTUNNEL = False  #@param {type:"boolean"}
NGROK_TOKEN = "" #@param {type:"string"}

import threading
import time
import socket

if NGROK_TOKEN != "":
  !pip -q install pyngrok
  from pyngrok import ngrok

  def iframe_thread(port):
    while True:
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', 8188))
        if result == 0:
          break
        sock.close()

    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(8188)
    print(tunnel.public_url)

  threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
  !python main.py --dont-print-server
elif USE_CLOUDFLARED:
  !wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
  !dpkg -i ~/cloudflared-linux-amd64.deb

  import subprocess
  import urllib.request

  def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
          break
        sock.close()
    print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
      l = line.decode()
      if "trycloudflare.com " in l:
        print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
      #print(l, end='')

  threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

  !python main.py --dont-print-server
elif USE_LOCALTUNNEL:
  !npm install -g localtunnel

  import subprocess
  import threading
  import time
  import socket
  def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
          break
        sock.close()
    print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)")
    p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
      print(line.decode(), end='')


  threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

  !python main.py --dont-print-server
else:
  from google.colab import output

  def iframe_thread(port):
      while True:
          time.sleep(0.5)
          sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
          result = sock.connect_ex(('127.0.0.1', port))
          if result == 0:
              break
          sock.close()
      print("Opening ComfyUI in iframe")
      output.serve_kernel_port_as_iframe(port, width='100%', height='800')

  threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
  !python main.py --dont-print-server
